# 3. Function Approximators for High-Dimensional State Spaces

## 3.1 The Curse of Dimensionality

In Chapter 4, we introduced tabular reinforcement learning methods where Q-values or policies were stored explicitly for each state-action pair. While this approach works well for small state spaces, it becomes computationally intractable when dealing with high-dimensional state spaces typical in topology optimization.

### 3.1.1 Limitations of Tabular Methods

Consider a topology optimization problem with:
- **Grid size**: 50 × 50 pixels
- **Material choices**: 3 (air, material 1, material 2)
- **State space size**: 3^2500 ≈ 10^1195 possible states

This astronomical number of states makes it impossible to:
1. Store Q-values for all state-action pairs
2. Visit each state during training
3. Compute optimal policies using dynamic programming

### 3.1.2 The Need for Function Approximation

Function approximation addresses the curse of dimensionality by learning a parameterized function that generalizes across similar states:

$$
Q(s, a; \theta) \approx Q^*(s, a)
$$

where $\theta$ represents the parameters of the approximation function.

**Key advantages**:
- **Generalization**: Learn from similar states without visiting all states
- **Compact representation**: Store millions of Q-values using thousands of parameters
- **Scalability**: Handle continuous and high-dimensional state spaces
- **Transfer learning**: Leverage knowledge from related tasks

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

# Visualize state space explosion
grid_sizes = np.arange(5, 51, 5)
materials = 3
state_space_sizes = materials ** (grid_sizes ** 2)

plt.figure(figsize=(12, 5))

# Plot 1: State space growth
plt.subplot(1, 2, 1)
plt.semilogy(grid_sizes, state_space_sizes, 'o-', linewidth=2, markersize=8)
plt.xlabel('Grid Size (N x N)', fontsize=12)
plt.ylabel('Number of States (log scale)', fontsize=12)
plt.title('Exponential Growth of State Space', fontsize=14)
plt.grid(True, alpha=0.3)

# Plot 2: Memory requirements
plt.subplot(1, 2, 2)
bytes_per_state = 8  # 8 bytes for double precision float
memory_requirements = state_space_sizes * bytes_per_state / (1024**3)  # GB
plt.semilogy(grid_sizes, memory_requirements, 's-', color='red', linewidth=2, markersize=8)
plt.xlabel('Grid Size (N x N)', fontsize=12)
plt.ylabel('Memory Required (GB, log scale)', fontsize=12)
plt.title('Memory Requirements for Tabular Methods', fontsize=14)
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"For a 50x50 grid with 3 materials:")
print(f"State space size: 3^2500 ≈ 10^{int(np.log10(3**2500))} states")
print(f"This is larger than the number of atoms in the observable universe!")

## 3.2 Linear Function Approximation

The simplest form of function approximation uses linear combinations of features:

$$
Q(s, a; \theta) = \theta^T \phi(s, a) = \sum_{i=1}^{n} \theta_i \phi_i(s, a)
$$

where:
- $\phi(s, a)$ is a feature vector extracted from state $s$ and action $a$
- $\theta$ is a weight vector with the same dimensionality as $\phi$

### 3.2.1 Feature Engineering

The success of linear function approximation depends critically on the quality of features:

**Common feature types:**
1. **Polynomial features**: $\phi_i(s) = s_1^{a_1} s_2^{a_2} \cdots s_n^{a_n}$
2. **Radial basis functions (RBF)**: $\phi_i(s) = \exp\left(-\frac{\|s - c_i\|^2}{2\sigma^2}\right)$
3. **Tile coding**: Partition state space into overlapping tiles
4. **Fourier basis**: Periodic features for continuous spaces

### 3.2.2 Gradient Descent Update

Parameters are updated using stochastic gradient descent:

$$
\theta \leftarrow \theta + \alpha \left[r + \gamma \max_{a'} Q(s', a'; \theta) - Q(s, a; \theta)\right] \nabla_\theta Q(s, a; \theta)
$$

For linear approximation:

$$
\nabla_\theta Q(s, a; \theta) = \phi(s, a)
$$

### 3.2.3 Advantages and Limitations

**Advantages:**
- Simple to implement and understand
- Well-understood convergence properties
- Computationally efficient
- No risk of overfitting with proper regularization

**Limitations:**
- Requires manual feature engineering
- Limited representational capacity
- Cannot learn complex non-linear relationships
- Performance depends on feature quality

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import PolynomialFeatures

class LinearQApproximator:
    """Linear Q-function approximator with polynomial features"""
    
    def __init__(self, state_dim: int, n_actions: int, degree: int = 2):
        self.state_dim = state_dim
        self.n_actions = n_actions
        self.degree = degree
        
        # Create polynomial feature transformer
        self.poly = PolynomialFeatures(degree=degree, include_bias=True)
        
        # Initialize with dummy state to determine feature dimension
        dummy_state = np.zeros(state_dim)
        dummy_features = self.poly.fit_transform(dummy_state.reshape(1, -1))
        self.feature_dim = dummy_features.shape[1]
        
        # Initialize weights for each action
        self.weights = np.zeros((n_actions, self.feature_dim))
    
    def get_features(self, state: np.ndarray) -> np.ndarray:
        """Extract polynomial features from state"""
        if state.ndim == 1:
            state = state.reshape(1, -1)
        return self.poly.transform(state)
    
    def predict(self, state: np.ndarray, action: int = None) -> np.ndarray:
        """Predict Q-value(s) for given state"""
        features = self.get_features(state)
        q_values = features @ self.weights.T
        
        if action is not None:
            return q_values[0, action]
        return q_values.squeeze()
    
    def update(self, state: np.ndarray, action: int, target: float, lr: float = 0.01):
        """Update weights using gradient descent"""
        features = self.get_features(state).squeeze()
        prediction = self.predict(state, action)
        error = target - prediction
        
        # Gradient descent update
        self.weights[action] += lr * error * features
        
        return error

# Demonstrate linear function approximation
np.random.seed(42)

# Create a simple 2D state space
state_dim = 2
n_actions = 4
approximator = LinearQApproximator(state_dim, n_actions, degree=2)

# Generate training data (simple reward function)
n_samples = 1000
states = np.random.uniform(-1, 1, (n_samples, state_dim))
actions = np.random.randint(0, n_actions, n_samples)

# True Q-function: rewards decrease with distance from origin
true_q = -np.linalg.norm(states, axis=1) + np.random.randn(n_samples) * 0.1

# Train the approximator
errors = []
for epoch in range(50):
    epoch_errors = []
    for state, action, target in zip(states, actions, true_q):
        error = approximator.update(state, action, target, lr=0.01)
        epoch_errors.append(error**2)
    errors.append(np.mean(epoch_errors))

# Visualize results
fig = plt.figure(figsize=(15, 5))

# Plot 1: Learning curve
plt.subplot(1, 3, 1)
plt.plot(errors, linewidth=2)
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Mean Squared Error', fontsize=12)
plt.title('Linear Approximator Learning Curve', fontsize=14)
plt.grid(True, alpha=0.3)

# Plot 2: True Q-function
ax = fig.add_subplot(1, 3, 2, projection='3d')
X = np.linspace(-1, 1, 30)
Y = np.linspace(-1, 1, 30)
X, Y = np.meshgrid(X, Y)
Z_true = -np.sqrt(X**2 + Y**2)
ax.plot_surface(X, Y, Z_true, cmap='viridis', alpha=0.7)
ax.set_xlabel('State Dim 1')
ax.set_ylabel('State Dim 2')
ax.set_zlabel('Q-value')
ax.set_title('True Q-function', fontsize=14)

# Plot 3: Learned Q-function
ax = fig.add_subplot(1, 3, 3, projection='3d')
states_grid = np.column_stack([X.ravel(), Y.ravel()])
Z_pred = np.array([approximator.predict(s, action=0) for s in states_grid])
Z_pred = Z_pred.reshape(X.shape)
ax.plot_surface(X, Y, Z_pred, cmap='viridis', alpha=0.7)
ax.set_xlabel('State Dim 1')
ax.set_ylabel('State Dim 2')
ax.set_zlabel('Q-value')
ax.set_title('Learned Q-function (Linear)', fontsize=14)

plt.tight_layout()
plt.show()

print(f"Final MSE: {errors[-1]:.4f}")
print(f"Feature dimension: {approximator.feature_dim}")
print(f"Number of parameters: {approximator.weights.size}")

## 3.3 Neural Network Function Approximation

Neural networks provide a powerful alternative to linear function approximation by automatically learning hierarchical feature representations.

### 3.3.1 Universal Function Approximation

The **Universal Approximation Theorem** states that a feedforward neural network with a single hidden layer containing a finite number of neurons can approximate any continuous function on compact subsets of $\mathbb{R}^n$ to arbitrary accuracy [[Hornik et al. 1989]](#references).

For Q-function approximation:

$$
Q(s, a; \theta) = f_\theta(s, a)
$$

where $f_\theta$ is a neural network with parameters $\theta$.

### 3.3.2 Multi-Layer Perceptron (MLP) Architecture

A typical MLP for Q-function approximation consists of:

1. **Input layer**: State representation (and optionally action)
2. **Hidden layers**: Multiple layers with non-linear activations
3. **Output layer**: Q-values for all actions

**Forward propagation:**

$$
\begin{align*}
h_1 &= \sigma(W_1 s + b_1) \\
h_2 &= \sigma(W_2 h_1 + b_2) \\
&\vdots \\
Q(s, \cdot; \theta) &= W_L h_{L-1} + b_L
\end{align*}
$$

where $\sigma$ is a non-linear activation function (e.g., ReLU, tanh).

### 3.3.3 Backpropagation and Training

Neural networks are trained using backpropagation to minimize the temporal difference error:

**Loss function:**

$$
L(\theta) = \mathbb{E}\left[\left(r + \gamma \max_{a'} Q(s', a'; \theta^-) - Q(s, a; \theta)\right)^2\right]
$$

where $\theta^-$ represents the parameters of a target network (discussed in Section 4).

**Gradient computation:**

$$
\nabla_\theta L(\theta) = -\mathbb{E}\left[\left(r + \gamma \max_{a'} Q(s', a'; \theta^-) - Q(s, a; \theta)\right) \nabla_\theta Q(s, a; \theta)\right]
$$

### 3.3.4 Activation Functions

Common activation functions used in DRL:

1. **ReLU (Rectified Linear Unit)**: $\sigma(x) = \max(0, x)$
   - Most popular for hidden layers
   - Addresses vanishing gradient problem
   - Computationally efficient

2. **Leaky ReLU**: $\sigma(x) = \max(\alpha x, x)$ where $\alpha \ll 1$
   - Prevents "dead neurons"
   - Allows small gradient when $x < 0$

3. **ELU (Exponential Linear Unit)**: 
   $$
   \sigma(x) = \begin{cases}
   x & \text{if } x > 0 \\
   \alpha(e^x - 1) & \text{if } x \leq 0
   \end{cases}
   $$
   - Smooth activation
   - Self-normalizing properties

4. **Tanh**: $\sigma(x) = \tanh(x) = \frac{e^x - e^{-x}}{e^x + e^{-x}}$
   - Output range [-1, 1]
   - Zero-centered
   - Can suffer from vanishing gradients

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt

class MLPQNetwork(nn.Module):
    """Multi-Layer Perceptron for Q-function approximation"""
    
    def __init__(self, state_dim: int, n_actions: int, hidden_dims: list = [128, 128]):
        super(MLPQNetwork, self).__init__()
        
        layers = []
        input_dim = state_dim
        
        # Build hidden layers
        for hidden_dim in hidden_dims:
            layers.append(nn.Linear(input_dim, hidden_dim))
            layers.append(nn.ReLU())
            input_dim = hidden_dim
        
        # Output layer
        layers.append(nn.Linear(input_dim, n_actions))
        
        self.network = nn.Sequential(*layers)
    
    def forward(self, state: torch.Tensor) -> torch.Tensor:
        """Forward pass through network"""
        return self.network(state)
    
    def predict(self, state: np.ndarray) -> np.ndarray:
        """Predict Q-values for numpy state"""
        with torch.no_grad():
            state_tensor = torch.FloatTensor(state)
            if state_tensor.dim() == 1:
                state_tensor = state_tensor.unsqueeze(0)
            q_values = self.forward(state_tensor)
            return q_values.numpy().squeeze()

# Compare different activation functions
activations = {
    'ReLU': lambda x: np.maximum(0, x),
    'Leaky ReLU': lambda x: np.maximum(0.01 * x, x),
    'ELU': lambda x: np.where(x > 0, x, 1.0 * (np.exp(x) - 1)),
    'Tanh': lambda x: np.tanh(x)
}

x = np.linspace(-3, 3, 1000)

plt.figure(figsize=(15, 5))

# Plot activation functions
plt.subplot(1, 2, 1)
for name, func in activations.items():
    plt.plot(x, func(x), label=name, linewidth=2)
plt.xlabel('Input', fontsize=12)
plt.ylabel('Output', fontsize=12)
plt.title('Activation Functions', fontsize=14)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)

# Plot derivatives
plt.subplot(1, 2, 2)
dx = x[1] - x[0]
for name, func in activations.items():
    y = func(x)
    dy = np.gradient(y, dx)
    plt.plot(x, dy, label=name, linewidth=2)
plt.xlabel('Input', fontsize=12)
plt.ylabel('Gradient', fontsize=12)
plt.title('Activation Function Gradients', fontsize=14)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Demonstrate MLP Q-network
state_dim = 10
n_actions = 4
q_network = MLPQNetwork(state_dim, n_actions, hidden_dims=[64, 64])

print("\nMLP Q-Network Architecture:")
print(q_network)
print(f"\nTotal parameters: {sum(p.numel() for p in q_network.parameters()):,}")

# Test forward pass
sample_state = np.random.randn(state_dim)
q_values = q_network.predict(sample_state)
print(f"\nSample state shape: {sample_state.shape}")
print(f"Q-values: {q_values}")
print(f"Best action: {np.argmax(q_values)}")

## 3.4 Convolutional Neural Networks for Spatial State Representations

For topology optimization, the state is naturally represented as a 2D grid (image) showing the material distribution. Convolutional Neural Networks (CNNs) are particularly well-suited for processing such spatial data.

### 3.4.1 CNN Architecture Principles

CNNs exploit three key properties:

1. **Local connectivity**: Neurons connect only to nearby neurons
2. **Parameter sharing**: Same filter applied across entire input
3. **Translation invariance**: Features detected regardless of position

### 3.4.2 Convolutional Layer

A convolutional layer applies learnable filters to the input:

$$
h_{ij}^{(l)} = \sigma\left(\sum_{m=0}^{k-1} \sum_{n=0}^{k-1} w_{mn}^{(l)} \cdot x_{(i+m)(j+n)}^{(l-1)} + b^{(l)}\right)
$$

where:
- $h_{ij}^{(l)}$ is the activation at position $(i,j)$ in layer $l$
- $w_{mn}^{(l)}$ is the filter weight at position $(m,n)$
- $k$ is the filter size
- $\sigma$ is the activation function

**Key hyperparameters:**
- **Kernel size**: Typically 3×3 or 5×5
- **Stride**: Step size for sliding the kernel
- **Padding**: Border handling (valid, same)
- **Number of filters**: Output channels

### 3.4.3 Pooling Layers

Pooling layers reduce spatial dimensions while retaining important features:

**Max pooling:**
$$
p_{ij} = \max_{m,n \in \mathcal{R}_{ij}} h_{mn}
$$

**Average pooling:**
$$
p_{ij} = \frac{1}{|\mathcal{R}_{ij}|} \sum_{m,n \in \mathcal{R}_{ij}} h_{mn}
$$

where $\mathcal{R}_{ij}$ is the pooling region.

### 3.4.4 CNN for Topology Optimization

A typical CNN architecture for processing topology states:

```
Input: [batch, channels, height, width]
  ↓
Conv2D(32, 3×3) + ReLU
  ↓
Conv2D(64, 3×3) + ReLU
  ↓
MaxPool2D(2×2)
  ↓
Conv2D(128, 3×3) + ReLU
  ↓
MaxPool2D(2×2)
  ↓
Flatten
  ↓
Dense(512) + ReLU
  ↓
Dense(n_actions)
```

### 3.4.5 Advantages for Topology Optimization

1. **Spatial structure preservation**: Maintains geometric relationships
2. **Translation invariance**: Learns patterns independent of location
3. **Parameter efficiency**: Fewer parameters than fully connected networks
4. **Hierarchical features**: Learns simple to complex patterns
5. **Scalability**: Handles different grid sizes with minimal changes

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from torchvision.utils import make_grid

class CNNQNetwork(nn.Module):
    """Convolutional Neural Network for topology optimization Q-function"""
    
    def __init__(self, input_channels: int, n_actions: int, grid_size: int = 50):
        super(CNNQNetwork, self).__init__()
        
        self.input_channels = input_channels
        self.n_actions = n_actions
        self.grid_size = grid_size
        
        # Convolutional layers
        self.conv1 = nn.Conv2d(input_channels, 32, kernel_size=3, stride=1, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1)
        
        # Calculate size after convolutions and pooling
        # Two max pooling layers with kernel size 2
        conv_output_size = grid_size // 4  # After two 2x2 max pooling
        self.flat_features = 128 * conv_output_size * conv_output_size
        
        # Fully connected layers
        self.fc1 = nn.Linear(self.flat_features, 512)
        self.fc2 = nn.Linear(512, n_actions)
    
    def forward(self, state: torch.Tensor) -> torch.Tensor:
        """Forward pass through CNN"""
        # Ensure correct input shape [batch, channels, height, width]
        if state.dim() == 3:
            state = state.unsqueeze(0)
        
        # Convolutional layers with ReLU and pooling
        x = F.relu(self.conv1(state))
        x = F.relu(self.conv2(x))
        x = F.max_pool2d(x, kernel_size=2)
        x = F.relu(self.conv3(x))
        x = F.max_pool2d(x, kernel_size=2)
        
        # Flatten and fully connected layers
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        q_values = self.fc2(x)
        
        return q_values
    
    def predict(self, state: np.ndarray) -> np.ndarray:
        """Predict Q-values for numpy state"""
        with torch.no_grad():
            state_tensor = torch.FloatTensor(state)
            q_values = self.forward(state_tensor)
            return q_values.numpy().squeeze()

# Create sample topology state
grid_size = 50
input_channels = 3  # Material, flux density, boundary
n_actions = grid_size * grid_size  # Can place material at any location

# Initialize CNN Q-network
cnn_q_network = CNNQNetwork(input_channels, n_actions, grid_size)

print("CNN Q-Network Architecture:")
print(cnn_q_network)
print(f"\nTotal parameters: {sum(p.numel() for p in cnn_q_network.parameters()):,}")

# Create sample state (material distribution + flux density + boundaries)
sample_state = np.random.rand(input_channels, grid_size, grid_size).astype(np.float32)

# Forward pass
q_values = cnn_q_network.predict(sample_state)

print(f"\nSample state shape: {sample_state.shape}")
print(f"Q-values shape: {q_values.shape}")
print(f"Best action: {np.argmax(q_values)}")

# Visualize sample state and learned features
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# Plot input channels
channel_names = ['Material Distribution', 'Flux Density', 'Boundary Conditions']
for i in range(3):
    axes[0, i].imshow(sample_state[i], cmap='viridis')
    axes[0, i].set_title(channel_names[i], fontsize=12)
    axes[0, i].axis('off')

# Visualize first layer filters
with torch.no_grad():
    filters = cnn_q_network.conv1.weight.cpu()
    
    # Normalize filters for visualization
    filters_min = filters.min()
    filters_max = filters.max()
    filters_norm = (filters - filters_min) / (filters_max - filters_min)
    
    # Show first 9 filters (3x3 grid)
    for i in range(3):
        # Average across input channels for visualization
        filter_viz = filters_norm[i].mean(dim=0).numpy()
        axes[1, i].imshow(filter_viz, cmap='gray')
        axes[1, i].set_title(f'Conv1 Filter {i+1}', fontsize=12)
        axes[1, i].axis('off')

plt.tight_layout()
plt.show()

# Compare parameter counts: MLP vs CNN
mlp_params = state_dim * 128 + 128 * 128 + 128 * n_actions
cnn_params = sum(p.numel() for p in cnn_q_network.parameters())

print(f"\nParameter Comparison:")
print(f"MLP (for flattened 50x50x3 input): {50*50*3*128 + 128*128 + 128*2500:,} parameters")
print(f"CNN (same input): {cnn_params:,} parameters")
print(f"CNN is {(50*50*3*128 + 128*128 + 128*2500) / cnn_params:.1f}x more efficient!")

## 3.5 Recurrent Neural Networks for Sequential Decision Making

While not used in our final implementation, Recurrent Neural Networks (RNNs) can be valuable for topology optimization when:
1. The optimization has temporal dependencies
2. Partial observability requires memory of past states
3. Long-term planning is critical

### 3.5.1 LSTM Architecture

Long Short-Term Memory (LSTM) networks address the vanishing gradient problem in standard RNNs:

$$
\begin{align*}
f_t &= \sigma(W_f \cdot [h_{t-1}, x_t] + b_f) \quad \text{(forget gate)} \\
i_t &= \sigma(W_i \cdot [h_{t-1}, x_t] + b_i) \quad \text{(input gate)} \\
\tilde{C}_t &= \tanh(W_C \cdot [h_{t-1}, x_t] + b_C) \quad \text{(candidate)} \\
C_t &= f_t \odot C_{t-1} + i_t \odot \tilde{C}_t \quad \text{(cell state)} \\
o_t &= \sigma(W_o \cdot [h_{t-1}, x_t] + b_o) \quad \text{(output gate)} \\
h_t &= o_t \odot \tanh(C_t) \quad \text{(hidden state)}
\end{align*}
$$

### 3.5.2 Application to Topology Optimization

RNNs can be integrated with CNNs for topology optimization:

```
State Sequence: s_{t-k}, ..., s_{t-1}, s_t
     ↓           ↓           ↓         ↓
   CNN         CNN         CNN       CNN (shared weights)
     ↓           ↓           ↓         ↓
   LSTM ← LSTM ← LSTM ← LSTM
     ↓
   Dense → Q-values
```

**Benefits:**
- Captures temporal dependencies in optimization process
- Maintains memory of past actions and their consequences
- Handles partially observable states

**Drawbacks:**
- Increased computational cost
- More difficult to train
- May overfit to training sequences

## 3.6 Summary and Comparison

### 3.6.1 Function Approximator Comparison

| Method | Advantages | Disadvantages | Best For |
|--------|-----------|---------------|----------|
| **Linear** | Simple, fast, convergence guarantees | Limited expressiveness, requires feature engineering | Low-dimensional, well-understood domains |
| **MLP** | Universal approximation, automatic features | Requires many parameters for spatial data | Vector state representations |
| **CNN** | Parameter efficient, spatial structure, translation invariance | Requires 2D/3D state representation | Image-like states, topology optimization |
| **RNN/LSTM** | Temporal dependencies, memory of past | Slow training, prone to overfitting | Partially observable, sequential problems |

### 3.6.2 Key Takeaways

1. **Function approximation is essential** for scaling RL to high-dimensional state spaces

2. **CNNs are ideal for topology optimization** due to:
   - Natural 2D grid representation
   - Parameter efficiency
   - Translation invariance
   - Hierarchical feature learning

3. **Trade-offs exist** between:
   - Expressiveness and sample efficiency
   - Computational cost and accuracy
   - Generalization and specialization

4. **Architecture matters** - proper network design significantly impacts:
   - Learning speed
   - Final performance
   - Stability during training

### 3.6.3 Next Steps

In the following sections, we'll explore:
- **Section 4**: Deep Q-Learning algorithms that combine Q-learning with neural networks
- **Section 5**: Policy gradient methods that directly optimize the policy
- **Section 6**: Actor-Critic algorithms that combine value and policy learning
- **Section 7**: Application to SeqTO-v2 environment with CNN architectures

## References

- Hornik, K., Stinchcombe, M., & White, H. (1989). Multilayer feedforward networks are universal approximators. *Neural Networks*, 2(5), 359-366.
- LeCun, Y., Bengio, Y., & Hinton, G. (2015). Deep learning. *Nature*, 521(7553), 436-444.
- Goodfellow, I., Bengio, Y., & Courville, A. (2016). *Deep Learning*. MIT Press.
- Mnih, V., et al. (2015). Human-level control through deep reinforcement learning. *Nature*, 518(7540), 529-533.
- Sutton, R. S., & Barto, A. G. (2018). *Reinforcement Learning: An Introduction* (2nd ed.). MIT Press.